<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/Video%20Sync%20And%20Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Joint Angle Trial**
> MARC - Multidisciplinary AI Research Centre | UOP


## **🛠️ Setup OpenDatasetStore**

In [ ]:
%%capture
# @title Store Initialization
from google.colab import drive
import pandas as pd
import os
!pip install -q ultralytics opencv-python pandas scipy
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git

from open_dataset_store import OpenDatasetStore

dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"
store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local",
    entity_id_format="sub_{num:04d}",
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)

import plotly.io as pio
pio.renderers.default = "colab"


In [ ]:
store.summary()
store.list_entries(entry_type="experiments")

# Visual Sync Pipeline

In [ ]:
# @title Utilities
import cv2
import json
import os
import numpy as np
import pandas as pd
from scipy import signal
import plotly.graph_objects as go
from ultralytics import YOLO

# YOLO (COCO) 17 keypoint names
KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

# Initialize YOLO model once
yolo_model = YOLO('yolo11x-pose.pt')

def analyze_keypoints(video_path):
    """
    Analyzes a video file for keypoints, saves the results to a JSON file,
    and returns a flattened pandas DataFrame with readable column names.
    """
    cap = cv2.VideoCapture(video_path)
    all_frames_data = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = yolo_model(frame, verbose=False)
        frame_data = {"frame": frame_idx}

        if len(results) > 0 and results[0].keypoints is not None:
            kpts = results[0].keypoints.xy[0].cpu().numpy().tolist()
            scores = results[0].keypoints.conf[0].cpu().numpy().tolist()

            if len(kpts) == 17:
                for i, name in enumerate(KEYPOINT_NAMES):
                    frame_data[f"{name}_x"] = kpts[i][0]
                    frame_data[f"{name}_y"] = kpts[i][1]
                    frame_data[f"{name}_score"] = scores[i]
            else:
                # Fill with NaNs if keypoints are incomplete
                for name in KEYPOINT_NAMES:
                    frame_data[f"{name}_x"] = np.nan
                    frame_data[f"{name}_y"] = np.nan
                    frame_data[f"{name}_score"] = np.nan
        else:
            # Fill with NaNs if no detections
            for name in KEYPOINT_NAMES:
                frame_data[f"{name}_x"] = np.nan
                frame_data[f"{name}_y"] = np.nan
                frame_data[f"{name}_score"] = np.nan

        all_frames_data.append(frame_data)
        frame_idx += 1

    cap.release()

    df = pd.DataFrame(all_frames_data)

    # Save to JSON
    output_json_path = video_path.replace('.mp4', '_keypoints.json')
    df.to_json(output_json_path, orient='records')

    print(f"Keypoints saved to {output_json_path}")
    return df

def calculate_sync_lag(front_df, side_df, plot=False):
    """
    Calculates the frame lag of the side video compared to the front video.
    Positive lag means side video is behind (lags), negative means front video is behind.
    """
    def get_movement_signal(df):
        coords = []
        for _, row in df.iterrows():
            kpts = []
            has_valid_kpts = True
            for name in KEYPOINT_NAMES:
                x = row.get(f"{name}_x", np.nan)
                y = row.get(f"{name}_y", np.nan)
                if pd.isna(x) or pd.isna(y):
                    has_valid_kpts = False
                    break
                kpts.append([x, y])

            if has_valid_kpts:
                coords.append(kpts)
            else:
                # Duplicate previous frame to avoid spikes on missing data
                coords.append(coords[-1] if len(coords) > 0 else [[0, 0]] * 17)

        coords = np.array(coords)
        diffs = np.diff(coords, axis=0)
        return np.sum(np.linalg.norm(diffs, axis=2), axis=1)

    sig_front = get_movement_signal(front_df)
    sig_side = get_movement_signal(side_df)

    sig_front_norm = (sig_front - np.mean(sig_front)) / (np.std(sig_front) + 1e-8)
    sig_side_norm = (sig_side - np.mean(sig_side)) / (np.std(sig_side) + 1e-8)

    correlation = signal.correlate(sig_front_norm, sig_side_norm, mode='full')
    lags = signal.correlation_lags(len(sig_front_norm), len(sig_side_norm), mode='full')

    best_lag = lags[np.argmax(correlation)]

    if plot:
        # Original Signals
        fig1 = go.Figure()
        fig1.add_trace(go.Scatter(y=sig_front_norm, mode='lines', name='Front Movement', opacity=0.8))
        fig1.add_trace(go.Scatter(y=sig_side_norm, mode='lines', name='Side Movement', opacity=0.8))
        fig1.update_layout(title="Original Un-Synced Movement Signals", template="plotly_dark", height=400)
        fig1.show()

        # Synced Signals
        fig2 = go.Figure()
        if best_lag > 0:
            x1 = np.arange(len(sig_front))
            x2 = np.arange(len(sig_side)) + best_lag
        else:
            x1 = np.arange(len(sig_front)) - best_lag
            x2 = np.arange(len(sig_side))

        fig2.add_trace(go.Scatter(x=x1, y=sig_front_norm, mode='lines', name='Front Movement (Synced)', opacity=0.8))
        fig2.add_trace(go.Scatter(x=x2, y=sig_side_norm, mode='lines', name='Side Movement (Synced)', opacity=0.8))
        fig2.update_layout(title=f"Aligned Movement Signals (Side Lag: {best_lag} frames)", template="plotly_dark", height=400)
        fig2.show()

    return best_lag

def generate_annotated_video(video_path, keypoints_df):
    """
    Generates an annotated video with skeleton overlays and returns the new file path.
    """
    SKELETON_PAIRS = [
        (0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9),
        (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13),
        (13, 15), (12, 14), (14, 16)
    ]
    CONFIDENCE_THRESHOLD = 0.4

    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    out_path = video_path.replace('.mp4', '_annotated.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= len(keypoints_df):
            break

        row = keypoints_df.iloc[frame_idx]
        kpts = []
        scores = []
        has_valid_kpts = True

        for name in KEYPOINT_NAMES:
            x = row.get(f"{name}_x", np.nan)
            y = row.get(f"{name}_y", np.nan)
            s = row.get(f"{name}_score", np.nan)
            if pd.isna(x) or pd.isna(y):
                has_valid_kpts = False
                break
            kpts.append([x, y])
            scores.append(s)

        if has_valid_kpts and len(kpts) == 17:
            for p1, p2 in SKELETON_PAIRS:
                if scores[p1] >= CONFIDENCE_THRESHOLD and scores[p2] >= CONFIDENCE_THRESHOLD:
                    pt1 = (int(kpts[p1][0]), int(kpts[p1][1]))
                    pt2 = (int(kpts[p2][0]), int(kpts[p2][1]))
                    cv2.line(frame, pt1, pt2, (0, 255, 0), 2, lineType=cv2.LINE_AA)

            for i, (x, y) in enumerate(kpts):
                if scores[i] >= CONFIDENCE_THRESHOLD:
                    cv2.circle(frame, (int(x), int(y)), 4, (0, 0, 255), -1, lineType=cv2.LINE_AA)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f"Annotated video saved to {out_path}")
    return out_path

def generate_synced_video(front_video_path, side_video_path, side_lag_frames):
    """
    Generates a synced side-by-side video based on frame lag without annotations.
    """
    cap1 = cv2.VideoCapture(front_video_path)
    cap2 = cv2.VideoCapture(side_video_path)

    if side_lag_frames > 0:
        cap1.set(cv2.CAP_PROP_POS_FRAMES, side_lag_frames)
    elif side_lag_frames < 0:
        cap2.set(cv2.CAP_PROP_POS_FRAMES, -side_lag_frames)

    fps = min(cap1.get(cv2.CAP_PROP_FPS), cap2.get(cv2.CAP_PROP_FPS))
    if fps == 0: fps = 30.0

    ret1, f1 = cap1.read()
    ret2, f2 = cap2.read()

    if ret1 and ret2:
        target_h = f1.shape[0]
        w1 = f1.shape[1]
        w2 = int(f2.shape[1] * (target_h / f2.shape[0]))
        out_w = w1 + w2

        # Reset to synced starting points
        cap1.set(cv2.CAP_PROP_POS_FRAMES, side_lag_frames if side_lag_frames > 0 else 0)
        cap2.set(cv2.CAP_PROP_POS_FRAMES, -side_lag_frames if side_lag_frames < 0 else 0)

        out_path = os.path.join(os.path.dirname(front_video_path), 'synced_side_by_side.mp4')
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(out_path, fourcc, fps, (out_w, target_h))

        while True:
            ret1, frame1 = cap1.read()
            ret2, frame2 = cap2.read()
            if not ret1 or not ret2:
                break

            if frame1.shape[0] != frame2.shape[0]:
                f2_resized = cv2.resize(frame2, (w2, target_h))
            else:
                f2_resized = frame2

            combined = np.hstack((frame1, f2_resized))
            out.write(combined)

        cap1.release()
        cap2.release()
        out.release()
        print(f"Synced video saved to {out_path}")
        return out_path
    else:
        print("Error: Could not read one or both videos.")
        return None

In [ ]:
# @title Visual Estimation Pipeline
import os
import glob

def visual_estimation_pipeline(entry_id, store, generate_annotated=True, generate_synced=True):
    """
    Pipeline for visual estimation of a specific experiment.

    Args:
        entry_id (str): The ID of the experiment to process.
        store (OpenDatasetStore): The initialized store object.
        generate_annotated (bool): Whether to generate skeleton-annotated videos.
        generate_synced (bool): Whether to generate a side-by-side synchronized video.

    Returns:
        tuple: (front_df, side_df, side_lag, front_annotated_path, side_annotated_path, synced_video_path)
    """
    entry_type = "experiments"
    labels = ["front_video", "side_video"]

    # Create experiment folder
    exp_folder = f"/content/{entry_id}"
    os.makedirs(exp_folder, exist_ok=True)
    print(f"Setting up directory: {exp_folder}")

    # Download videos
    for label in labels:
        print(f"Downloading {label}...")
        store.download_related_file(
            entry_type=entry_type,
            entry_id=entry_id,
            label=label,
            dest_folder=exp_folder
        )

    # Identify the downloaded files
    all_mp4s = glob.glob(f"{exp_folder}/*.mp4")
    front_video_path = next((f for f in all_mp4s if "front_video" in f), None)
    side_video_path = next((f for f in all_mp4s if "side_video" in f), None)

    if not front_video_path or not side_video_path:
        print("Error: Could not locate both downloaded video files.")
        return None, None, None, None, None, None

    print("\n--- Analyzing Keypoints ---")
    front_df = analyze_keypoints(front_video_path)
    side_df = analyze_keypoints(side_video_path)

    print("\n--- Calculating Sync Lag ---")
    side_lag = calculate_sync_lag(front_df, side_df, plot=True)
    print(f"Calculated side lag: {side_lag} frames")

    front_annotated = None
    side_annotated = None
    synced_video = None

    if generate_annotated:
        print("\n--- Generating Annotated Videos ---")
        front_annotated = generate_annotated_video(front_video_path, front_df)
        side_annotated = generate_annotated_video(side_video_path, side_df)

    if generate_synced:
        print("\n--- Generating Synchronized Video ---")
        synced_video = generate_synced_video(front_video_path, side_video_path, side_lag)

    print("\nPipeline complete!")
    return front_df, side_df, side_lag, front_annotated, side_annotated, synced_video

In [ ]:
# @title Save Results
def save_visual_estimation_results(entry_id, store, front_df, side_df, side_lag,
                                   front_annotated_path=None, side_annotated_path=None,
                                   synced_video_path=None):
    """
    Saves the results from the visual estimation pipeline back to the OpenDatasetStore.
    """
    entry_type = "experiments"
    print(f"--- Saving results for {entry_id} to OpenDatasetStore ---")

    # 1. Save Metadata (Lag Frames)
    if side_lag is not None:
        store.edit_entry(
            entry_type=entry_type,
            entry_id=entry_id,
            **{"video_sync_side_lag_frames": int(side_lag)}
        )
        print(f"✓ Saved metadata: video_sync_side_lag_frames = {int(side_lag)}")

    # 2. Save Keypoint DataFrames
    if front_df is not None and not front_df.empty:
        store.add_processed_data(
            entry_type=entry_type,
            entry_id=entry_id,
            tag="front_video_keypoints",
            df=front_df
        )
        print("✓ Saved front_video_keypoints dataframe")

    if side_df is not None and not side_df.empty:
        store.add_processed_data(
            entry_type=entry_type,
            entry_id=entry_id,
            tag="side_video_keypoints",
            df=side_df
        )
        print("✓ Saved side_video_keypoints dataframe")

    # 3. Save Generated Videos
    def save_video(file_path, label):
        if file_path and os.path.exists(file_path):
            try:
                # Assuming the store has an upload_related_file method that accepts file_path
                store.add_related_file(
                    entry_type=entry_type,
                    entry_id=entry_id,
                    label=label,
                    local_path=file_path
                )
                print(f"✓ Saved video file for label: {label}")
            except AttributeError:
                print(f"⚠ Could not automatically save {label}. Store API might require a different method (e.g. upload_related_file_interactive).")
            except Exception as e:
                print(f"⚠ Error saving {label}: {e}")

    save_video(synced_video_path, "synced_video")
    save_video(front_annotated_path, "front_annotated_video")
    save_video(side_annotated_path, "side_annotated_video")

    print("--- Save Complete! ---")


In [ ]:
# @title Run Full Pipeline

target_entry_id = "experiments_0001" #@param {type:"string"}

print(f"Starting end-to-end process for {target_entry_id}...")

# 1. Run the pipeline to generate DataFrames and videos
front_df, side_df, side_lag, front_annotated_path, side_annotated_path, synced_video_path = visual_estimation_pipeline(
    entry_id=target_entry_id,
    store=store,
    generate_annotated=True,
    generate_synced=True
)


In [ ]:
# @title Ensure data is valid, then save
if front_df is not None and side_df is not None:
    # 3. Save all results back to the OpenDatasetStore
    save_visual_estimation_results(
        entry_id=target_entry_id,
        store=store,
        front_df=front_df,
        side_df=side_df,
        side_lag=side_lag,
        front_annotated_path=front_annotated_path,
        side_annotated_path=side_annotated_path,
        synced_video_path=synced_video_path
    )
    print(f"\nSuccessfully completed processing and saving for {target_entry_id}!")
else:
    print(f"\nFailed to process {target_entry_id}. Please check the logs above for errors.")

# 3-D Pose Fusion

In [ ]:
# @title  3-D Pose Fusion  (pre-synced)
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.sparse import csr_matrix
import warnings
warnings.filterwarnings('ignore')

# ── COCO 17-keypoint skeleton ──────────────────────────────────────────
SKELETON_PAIRS = [
    (0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9),
    (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13),
    (13, 15), (12, 14), (14, 16)
]

# ── Load & sync once ───────────────────────────────────────────────────
experiment_id = 'experiments_0001'
frame_lag     = store.get_entry(
    entry_type="experiments", entry_id=experiment_id
)['video_sync_side_lag_frames']
frame_lag = -1 * frame_lag
fk_df_raw = store.get_entry_processed_data(
    entry_type="experiments", entry_id=experiment_id, tag="front_video_keypoints")
sk_df_raw = store.get_entry_processed_data(
    entry_type="experiments", entry_id=experiment_id, tag="side_video_keypoints")

# Apply lag so both sequences start at the same physical moment.
# Positive lag  → side video starts frame_lag frames later than front
#                 → drop the first frame_lag rows from side
# Negative lag  → front video starts |frame_lag| frames later than side
#                 → drop the first |frame_lag| rows from front
if frame_lag >= 0:
    fk_sync = fk_df_raw.iloc[:].reset_index(drop=True)
    sk_sync = sk_df_raw.iloc[frame_lag:].reset_index(drop=True)
else:
    fk_sync = fk_df_raw.iloc[-frame_lag:].reset_index(drop=True)
    sk_sync = sk_df_raw.iloc[:].reset_index(drop=True)

# Trim to the shorter so both are exactly the same length
n_synced = min(len(fk_sync), len(sk_sync))
fk_df    = fk_sync.iloc[:n_synced].reset_index(drop=True)
sk_df    = sk_sync.iloc[:n_synced].reset_index(drop=True)

print(f"frame_lag = {frame_lag}  →  {n_synced} synced frames "
      f"(front: {len(fk_df_raw)}, side: {len(sk_df_raw)})")


# ── Bayesian MAP 3-D fuser ─────────────────────────────────────────────
class Bayesian3DFuser:
    """
    Batch Bayesian MAP estimator fusing two 2D YOLO pose streams
    (front + side) into a single 3D skeleton with confidences and
    one consistent set of bone lengths.

    Coordinate convention (after root-centering and scale normalisation)
    -------------------------------------------------------------------
    Front camera observes  (X, Y)  →  world  X = lateral, Y = vertical
    Side  camera observes  (Z, Y)  →  world  Z = depth,   Y = vertical

    Model
    -----
    X_t  : 3D joints at frame t          (latent)
    L_b  : global bone length for bone b (latent, shared over frames)

    MAP objective:
        Σ c·‖proj(X) – y‖²               (weighted reprojection)
      + λ_bone · Σ (‖X_p – X_c‖ – L_b)² (bone-length prior)
      + λ_temp · Σ ‖X_t – 2X_{t-1} + X_{t-2}‖²  (smoothness)
      Bone lengths updated via EM (median across frames).
    """

    def __init__(self,
                 keypoint_names,
                 skeleton_pairs=SKELETON_PAIRS,
                 temporal_weight=1.0,
                 bone_weight=10.0,
                 robust_scale=5.0,
                 n_em_iters=3):
        self.names          = list(keypoint_names)
        self.pairs          = list(skeleton_pairs)
        self.n_joints       = len(self.names)
        self.temporal_weight = temporal_weight
        self.bone_weight    = bone_weight
        self.robust_scale   = robust_scale
        self.n_em_iters     = n_em_iters

        self.root_left  = 11   # left  hip  (COCO 17)
        self.root_right = 12   # right hip

        self._scale_ref_pairs = [(11, 13), (12, 14), (13, 15), (14, 16)]

        # set by fuse_sequence(); used by the visualisation
        self.scale_f     = 1.0
        self.scale_s     = 1.0
        self.front_roots = None
        self.side_roots  = None

    # ── data extraction ────────────────────────────────────────────────
    def _extract_all(self, df, num_frames):
        J   = self.n_joints
        pts = np.full((num_frames, J, 2), np.nan)
        sc  = np.zeros((num_frames, J))
        for t in range(num_frames):
            row = df.iloc[t]
            for j, name in enumerate(self.names):
                pts[t, j, 0] = row.get(f"{name}_x",     np.nan)
                pts[t, j, 1] = row.get(f"{name}_y",     np.nan)
                sc [t, j]    = float(row.get(f"{name}_score", 0.0))
        return pts, sc

    # ── root centring ──────────────────────────────────────────────────
    def _center_all(self, pts, scores):
        T       = pts.shape[0]
        centered = pts.copy()
        roots   = np.zeros((T, 2))
        for t in range(T):
            if (scores[t, self.root_left]  > 0.1
                    and scores[t, self.root_right] > 0.1):
                root = (pts[t, self.root_left] + pts[t, self.root_right]) / 2.0
            else:
                valid = pts[t][scores[t] > 0.1]
                root  = (np.nanmean(valid, axis=0)
                         if len(valid) > 0 else np.array([0.0, 0.0]))
            centered[t] = pts[t] - root
            roots[t]    = root
        return centered, roots

    # ── per-view scale normalisation ───────────────────────────────────
    def _estimate_scale(self, pts, scores):
        lengths = []
        for t in range(pts.shape[0]):
            for (a, b) in self._scale_ref_pairs:
                if (scores[t, a] > 0.2 and scores[t, b] > 0.2
                        and not np.isnan(pts[t, a, 0])
                        and not np.isnan(pts[t, b, 0])):
                    d = np.linalg.norm(pts[t, a] - pts[t, b])
                    if d > 1e-6:
                        lengths.append(d)
        return 1.0 / float(np.median(lengths)) if lengths else 1.0

    # ── bone-length initialisation ─────────────────────────────────────
    def _init_bone_lengths(self, f_pts, s_pts, f_valid, s_valid):
        L = {}
        for (p1, p2) in self.pairs:
            f_est, s_est = [], []
            for t in range(f_pts.shape[0]):
                if f_valid[t, p1] and f_valid[t, p2]:
                    f_est.append(np.linalg.norm(f_pts[t, p1] - f_pts[t, p2]))
            for t in range(s_pts.shape[0]):
                if s_valid[t, p1] and s_valid[t, p2]:
                    s_est.append(np.linalg.norm(s_pts[t, p1] - s_pts[t, p2]))
            cand = []
            if f_est: cand.append(np.median(f_est))
            if s_est: cand.append(np.median(s_est))
            L[(p1, p2)] = float(max(cand)) if cand else 1.0
        return L

    # ── Jacobian sparsity ──────────────────────────────────────────────
    def _build_sparsity(self, n_frames):
        J = self.n_joints
        P = len(self.pairs)

        n_reproj = 4 * n_frames * J
        n_motion = 3 * max(0, n_frames - 2) * J
        n_bone   = n_frames * P
        n_res    = n_reproj + n_motion + n_bone
        n_par    = n_frames * J * 3

        rows, cols = [], []
        r = 0

        def add(row, col):
            rows.append(row); cols.append(col)

        # Front-X
        for t in range(n_frames):
            for j in range(J):
                add(r, t * J * 3 + j * 3 + 0); r += 1
        # Front-Y
        for t in range(n_frames):
            for j in range(J):
                add(r, t * J * 3 + j * 3 + 1); r += 1
        # Side-Z
        for t in range(n_frames):
            for j in range(J):
                add(r, t * J * 3 + j * 3 + 2); r += 1
        # Side-Y
        for t in range(n_frames):
            for j in range(J):
                add(r, t * J * 3 + j * 3 + 1); r += 1

        # Temporal smoothness
        for t in range(2, n_frames):
            for j in range(J):
                for c in range(3):
                    add(r, t       * J * 3 + j * 3 + c)
                    add(r, (t - 1) * J * 3 + j * 3 + c)
                    add(r, (t - 2) * J * 3 + j * 3 + c)
                    r += 1

        # Bone-length
        for t in range(n_frames):
            for (p1, p2) in self.pairs:
                for c in range(3):
                    add(r, t * J * 3 + p1 * 3 + c)
                    add(r, t * J * 3 + p2 * 3 + c)
                r += 1

        assert r == n_res, f"Sparsity row count mismatch: {r} vs {n_res}"
        data = np.ones(len(rows), dtype=bool)
        return csr_matrix((data, (rows, cols)), shape=(n_res, n_par))

    # ── one batch solve (E-step) ───────────────────────────────────────
    def _solve_poses(self, X_init, f_pts, s_pts, f_scores, s_scores,
                     f_valid, s_valid, bone_lengths, jac_sparsity):
        n_frames = f_pts.shape[0]
        J        = self.n_joints
        sw_bone  = float(np.sqrt(self.bone_weight))
        sw_temp  = float(np.sqrt(self.temporal_weight))

        w_f = np.sqrt(np.maximum(f_scores, 0.0)) * f_valid
        w_s = np.sqrt(np.maximum(s_scores, 0.0)) * s_valid

        def residuals(flat):
            X   = flat.reshape(n_frames, J, 3)
            out = []

            out.append(((X[:, :, 0] - f_pts[:, :, 0]) * w_f).ravel())
            out.append(((X[:, :, 1] - f_pts[:, :, 1]) * w_f).ravel())
            out.append(((X[:, :, 2] - s_pts[:, :, 0]) * w_s).ravel())
            out.append(((X[:, :, 1] - s_pts[:, :, 1]) * w_s).ravel())

            if n_frames >= 3:
                acc = (X[2:] - 2.0 * X[1:-1] + X[:-2]) * sw_temp
                out.append(acc.ravel())

            bone_res = np.empty((n_frames, len(self.pairs)))
            for bi, (p1, p2) in enumerate(self.pairs):
                d = np.linalg.norm(X[:, p1] - X[:, p2], axis=1)
                bone_res[:, bi] = (d - bone_lengths[(p1, p2)]) * sw_bone
            out.append(bone_res.ravel())

            return np.concatenate(out)

        result = least_squares(
            residuals, X_init,
            jac_sparsity=jac_sparsity,
            method='trf',
            loss='soft_l1',
            f_scale=self.robust_scale,
            x_scale='jac',
            max_nfev=200,
            verbose=0,
        )
        return result.x

    # ── main API  (no lag — caller pre-syncs) ─────────────────────────
    def fuse_sequence(self, front_df, side_df, num_frames=None):
        """
        front_df and side_df must already be temporally aligned
        (same number of rows, row i in both corresponds to the same instant).
        """
        available = min(len(front_df), len(side_df))
        num_frames = available if num_frames is None else min(num_frames, available)
        if num_frames < 3:
            raise ValueError("Need at least 3 frames for temporal smoothing.")

        f_slice = front_df.iloc[:num_frames]
        s_slice = side_df.iloc[:num_frames]

        f_pts_raw, f_scores = self._extract_all(f_slice, num_frames)
        s_pts_raw, s_scores = self._extract_all(s_slice, num_frames)

        # 1) root-centre each view
        f_pts, front_roots = self._center_all(f_pts_raw, f_scores)
        s_pts, side_roots  = self._center_all(s_pts_raw, s_scores)

        # 2) per-view scale normalisation
        scale_f = self._estimate_scale(f_pts, f_scores)
        scale_s = self._estimate_scale(s_pts, s_scores)
        f_pts   = f_pts * scale_f
        s_pts   = s_pts * scale_s

        # 3) validity masks
        f_valid = (~np.isnan(f_pts[:, :, 0])) & (f_scores > 0.05)
        s_valid = (~np.isnan(s_pts[:, :, 0])) & (s_scores > 0.05)

        f_clean = np.where(np.isnan(f_pts), 0.0, f_pts)
        s_clean = np.where(np.isnan(s_pts), 0.0, s_pts)

        # 4) initial 3-D guess (validity-weighted Y blend)
        X0 = np.zeros((num_frames, self.n_joints, 3))
        X0[:, :, 0] = f_clean[:, :, 0]
        X0[:, :, 2] = s_clean[:, :, 0]

        f_w      = f_valid.astype(float)
        s_w      = s_valid.astype(float)
        denom    = f_w + s_w
        denom_s  = np.where(denom > 0, denom, 1.0)
        X0[:, :, 1] = (f_clean[:, :, 1] * f_w
                       + s_clean[:, :, 1] * s_w) / denom_s

        X_flat = X0.ravel()

        # 5) initial bone lengths
        bone_lengths = self._init_bone_lengths(f_clean, s_clean, f_valid, s_valid)

        # 6) sparsity pattern
        jac_sp = self._build_sparsity(num_frames)

        # 7) EM loop
        X_opt = X0.copy()
        for em in range(self.n_em_iters):
            X_flat = self._solve_poses(
                X_flat,
                f_clean, s_clean, f_scores, s_scores,
                f_valid, s_valid, bone_lengths, jac_sp,
            )
            X_opt = X_flat.reshape(num_frames, self.n_joints, 3)

            new_L = {}
            for (p1, p2) in self.pairs:
                d = np.linalg.norm(X_opt[:, p1] - X_opt[:, p2], axis=1)
                new_L[(p1, p2)] = float(np.median(d))

            delta = max(abs(new_L[k] - bone_lengths[k]) for k in new_L)
            bone_lengths = new_L
            print(f"  EM iter {em + 1}/{self.n_em_iters}  |  max |ΔL| = {delta:.4f}")

        # 8) fused confidence
        seen_any     = f_valid | s_valid
        fused_scores = np.where(
            seen_any, np.clip(f_scores + s_scores, 0.0, 1.0), 0.0)

        # 9) convert back to front-view pixel units
        inv_scale_f = 1.0 / scale_f
        X_out       = X_opt * inv_scale_f
        bone_arr    = np.array(
            [bone_lengths[(p1, p2)] * inv_scale_f for (p1, p2) in self.pairs])

        # store state needed by the visualisation
        self.scale_f     = scale_f
        self.scale_s     = scale_s
        self.front_roots = front_roots
        self.side_roots  = side_roots

        return X_out, fused_scores, bone_arr


# ── Run ────────────────────────────────────────────────────────────────
fuser = Bayesian3DFuser(
    KEYPOINT_NAMES,
    SKELETON_PAIRS,
    temporal_weight=1.0,
    bone_weight=10.0,
    robust_scale=5.0,
    n_em_iters=3,
)

fused_3d_pts, fused_scores, final_bones = fuser.fuse_sequence(
    fk_df, sk_df,          # already synced — no lag argument needed
    num_frames=len(fk_df),
)

print("\n─── 3-D Fusion Summary ───")
print("Output tensor shape:", fused_3d_pts.shape)
print("Bone lengths (front-view pixels):")
for (p1, p2), L in zip(SKELETON_PAIRS, final_bones):
    print(f"  {KEYPOINT_NAMES[p1]:>12s} → {KEYPOINT_NAMES[p2]:<12s} : {L:7.2f}")

In [ ]:
# @title  2-D Pose Viewer  (front + side)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

num_frames = fused_3d_pts.shape[0]
J          = len(KEYPOINT_NAMES)

# ── 1. Raw 2-D poses ───────────────────────────────────────────────────
def extract_2d(df, n):
    pts = np.full((n, J, 2), np.nan)
    sc  = np.zeros((n, J))
    for t in range(n):
        row = df.iloc[t]
        for j, name in enumerate(KEYPOINT_NAMES):
            pts[t, j, 0] = row.get(f"{name}_x",  np.nan)
            pts[t, j, 1] = row.get(f"{name}_y",  np.nan)
            sc [t, j]    = float(row.get(f"{name}_score", 0.0))
    return pts, sc

f_raw, f_raw_sc = extract_2d(fk_df, num_frames)
s_raw, s_raw_sc = extract_2d(sk_df, num_frames)

# ── 2. Reprojections ───────────────────────────────────────────────────
ratio_fs = fuser.scale_f / fuser.scale_s

def reproject_front(pts3d, root):
    out = np.empty((J, 2))
    out[:, 0] = pts3d[:, 0] + root[0]
    out[:, 1] = pts3d[:, 1] + root[1]
    return out

def reproject_side(pts3d, root):
    out = np.empty((J, 2))
    out[:, 0] = pts3d[:, 2] * ratio_fs + root[0]
    out[:, 1] = pts3d[:, 1] * ratio_fs + root[1]
    return out

f_reproj = np.stack([reproject_front(fused_3d_pts[t], fuser.front_roots[t])
                     for t in range(num_frames)])
s_reproj = np.stack([reproject_side (fused_3d_pts[t], fuser.side_roots[t])
                     for t in range(num_frames)])

# ── 3. Axis ranges (fixed over all frames) ─────────────────────────────
def range_2d(*sets):
    xmin = min(np.nanmin(p[:, :, 0]) for p in sets)
    xmax = max(np.nanmax(p[:, :, 0]) for p in sets)
    ymin = min(np.nanmin(p[:, :, 1]) for p in sets)
    ymax = max(np.nanmax(p[:, :, 1]) for p in sets)
    padx = 0.10 * (xmax - xmin) + 10
    pady = 0.10 * (ymax - ymin) + 10
    return [xmin - padx, xmax + padx], [ymax + pady, ymin - pady]

f_xr, f_yr = range_2d(f_raw, f_reproj)
s_xr, s_yr = range_2d(s_raw, s_reproj)

# ── 4. Trace builders ──────────────────────────────────────────────────
def _bones_xy(pts):
    bx, by = [], []
    for p1, p2 in SKELETON_PAIRS:
        bx.extend([pts[p1, 0], pts[p2, 0], None])
        by.extend([pts[p1, 1], pts[p2, 1], None])
    return bx, by

def raw_traces(pts, sc):
    bx, by = _bones_xy(pts)
    return [
        go.Scatter(x=bx, y=by, mode='lines',
                   line=dict(color='rgba(180,180,180,0.50)', width=2),
                   hoverinfo='none', showlegend=False),
        go.Scatter(x=pts[:, 0], y=pts[:, 1], mode='markers',
                   marker=dict(size=sc * 10 + 2,
                               color='rgba(180,180,180,0.50)',
                               line=dict(width=1, color='rgba(255,255,255,0.3)')),
                   text=[f"RAW {KEYPOINT_NAMES[j]}<br>{sc[j]:.2f}"
                         for j in range(J)],
                   hoverinfo='text', showlegend=False),
    ]

def est_traces(pts, sc):
    bx, by = _bones_xy(pts)
    return [
        go.Scatter(x=bx, y=by, mode='lines',
                   line=dict(color='cyan', width=3),
                   hoverinfo='none', showlegend=False),
        go.Scatter(x=pts[:, 0], y=pts[:, 1], mode='markers',
                   marker=dict(size=sc * 15 + 3,
                               color=sc, colorscale='Plasma',
                               cmin=0, cmax=1,
                               line=dict(width=1, color='white')),
                   text=[f"EST {KEYPOINT_NAMES[j]}<br>{sc[j]:.2f}"
                         for j in range(J)],
                   hoverinfo='text', showlegend=False),
    ]

# ── 5. Figure ──────────────────────────────────────────────────────────
# Trace index map (8 traces total):
#   0,1  front raw  bones/joints
#   2,3  front est  bones/joints
#   4,5  side  raw  bones/joints
#   6,7  side  est  bones/joints

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Front  —  raw (grey) vs reprojected (cyan)",
                    "Side   —  raw (grey) vs reprojected (cyan)"),
    column_widths=[0.5, 0.5],
    horizontal_spacing=0.06,
)

init = (
    raw_traces(f_raw[0],    f_raw_sc[0])    +   # 0, 1
    est_traces(f_reproj[0], fused_scores[0]) +   # 2, 3
    raw_traces(s_raw[0],    s_raw_sc[0])    +   # 4, 5
    est_traces(s_reproj[0], fused_scores[0])     # 6, 7
)
for tr in init[0:4]: fig.add_trace(tr, row=1, col=1)
for tr in init[4:8]: fig.add_trace(tr, row=1, col=2)

# ── 6. Animation frames ────────────────────────────────────────────────
fig.frames = [
    go.Frame(
        data=(
            raw_traces(f_raw[i],    f_raw_sc[i])    +
            est_traces(f_reproj[i], fused_scores[i]) +
            raw_traces(s_raw[i],    s_raw_sc[i])    +
            est_traces(s_reproj[i], fused_scores[i])
        ),
        name=str(i),
        traces=[0, 1, 2, 3, 4, 5, 6, 7],
    )
    for i in range(num_frames)
]

# ── 7. Layout ──────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=f"2-D Pose Viewer  |  {num_frames} frames  |  lag {frame_lag}",
        x=0.5, xanchor='center', font=dict(size=15),
    ),
    template='plotly_dark',
    height=700,
    margin=dict(l=10, r=10, t=70, b=80),
    showlegend=False,

    xaxis =dict(range=f_xr, autorange=False, title='u  (front)',
                scaleanchor='y',  scaleratio=1),
    yaxis =dict(range=f_yr, autorange=False, title='v  (front)'),
    xaxis2=dict(range=s_xr, autorange=False, title='u  (side)',
                scaleanchor='y2', scaleratio=1),
    yaxis2=dict(range=s_yr, autorange=False, title='v  (side)'),

    updatemenus=[dict(
        type='buttons', showactive=False,
        x=0.5, y=-0.08, xanchor='center',
        buttons=[
            dict(label='▶  Play', method='animate',
                 args=[None, dict(
                     frame=dict(duration=60, redraw=False),
                     fromcurrent=True,
                     transition=dict(duration=0),
                 )]),
            dict(label='⏸  Pause', method='animate',
                 args=[[None], dict(
                     frame=dict(duration=0, redraw=False),
                     mode='immediate',
                     transition=dict(duration=0),
                 )]),
        ],
    )],
    sliders=[dict(
        steps=[
            dict(method='animate',
                 args=[[str(i)],
                       dict(mode='immediate',
                            frame=dict(duration=0, redraw=False),
                            transition=dict(duration=0))],
                 label=str(i))
            for i in range(num_frames)
        ],
        transition=dict(duration=0),
        x=0.0, y=-0.02, len=1.0,
        currentvalue=dict(font=dict(size=12), prefix='Frame: ',
                          visible=True, xanchor='right'),
        pad=dict(t=10),
    )],
)

fig.show()